### Transform Races Data

In [0]:
%run ../00-common/01-environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

#### 1. Read Races data

In [0]:
races_df = spark.table(bronze_table) 

display(races_df)

#### 2. Remove NULL VALUES

In [0]:
from pyspark.sql import functions as F

races_valid_df = races_df.filter(F.col('circuitId').isNotNull())

display(races_valid_df)

#### 3. Kepp columns that are needed 

In [0]:
races_selected_df = races_df.select(
    F.col('season'),
    F.col('round'),
    F.col('raceName'),
    F.col('date'),
    F.col('circuitId'),
    F.col('file_path'),
    F.col('ingestion_timestamp')
)

display(races_selected_df)

#### 4. Rename Columns

In [0]:
races_renamed_df = races_selected_df.withColumnsRenamed({
    'circuitId': 'circuit_id',
    'raceName': 'race_name'
})

display(races_renamed_df)


#### 5. Check for duplicates and Remove if needed

In [0]:
races_distinct_df = races_renamed_df.distinct()

In [0]:
display(races_distinct_df)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates([ 'round', 'season'])
display(races_distinct_df)

### 6. Transform Values of Columns race_name to Title Case

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn('race_name', F.initcap(F.col('race_name')))
    )

display(races_final_df)

### 7. Writing data to bronze table

In [0]:
(
    races_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))